# Dashboard Dataset Preparation

## Objective

Prepare dashboard-ready datasets for Supply Chain Control Tower reporting and executive monitoring.

This notebook aggregates operational KPIs from Capacity Risk, Inventory Risk, and Service Risk modules into reporting datasets optimized for dashboard visualization.

## Data Sources

- capacity_utilization_weekly.csv
- inventory_risk_weekly.csv
- service_risk_weekly.csv
- top_product_driver_weekly.csv

## Deliverables

- dashboard_capacity_summary.csv
- dashboard_inventory_summary.csv
- dashboard_service_summary.csv
- executive_dashboard_summary.csv

## Dashboard Goals

Provide visibility into:

- Capacity utilization trends
- Inventory health and stockout risk
- Service risk exposure
- Root cause drivers
- Executive KPI summary

In [1]:
import pandas as pd
import numpy as np


In [2]:
capacity_utilization = pd.read_csv(
    "../data/processed/capacity_utilization_weekly.csv"
)

inventory_risk = pd.read_csv(
    "../data/processed/inventory_risk_weekly.csv"
)

service_risk = pd.read_csv(
    "../data/processed/service_risk_weekly.csv"
)

top_product_driver = pd.read_csv(
    "../data/processed/top_product_driver_weekly.csv"
)

### ==========================================
### Dashboard Capacity Summary
### ==========================================

In [9]:
dashboard_service_summary = (
    service_risk[
        [
            "Plant_ID",
            "Product Card Id",
            "Product Name",
            "Order_Date",
            "Service_Risk",
            "Service_Risk_Score",
            "Service_Alert",
            "Risk_Driver"
        ]
    ]
    .copy()
)


In [ ]:
#dashboard_capacity_summary.to_csv(
#    "../data/processed/dashboard_capacity_summary.csv",
#    index=False)

### ==========================================
### Dashboard Inventory Summary
### ==========================================

In [21]:
dashboard_inventory_summary = (
    inventory_risk[
        [
            "Plant_ID",
            "Product Card Id",
            "Product Name",
            "Order_Date",
            "Inventory_On_Hand",
            "DOI",
            "Inventory_Status",
            "Inventory_Alert",
            "Inventory_Utilization"
        ]
    ]
    .copy()
)

dashboard_inventory_summary.to_csv(
    "../data/processed/dashboard_inventory_summary.csv",
    index=False
)

### ==========================================
### Dashboard Service Summary
### ==========================================

In [22]:
dashboard_service_summary = (
    service_risk[
        [
            "Plant_ID",
            "Product Card Id",
            "Product Name",
            "Order_Date",
            "Service_Risk",
            "Service_Risk_Score",
            "Service_Alert",
            "Risk_Driver"
        ]
    ]
    .copy()
)

dashboard_service_summary.to_csv(
    "../data/processed/dashboard_service_summary.csv",
    index=False
)

In [11]:
# Capacity Alert
capacity_alert_weekly = (
    capacity_utilization
    .groupby("Order_Date")
    ["Capacity_Status"]
    .apply(
        lambda x:
        (x == "Critical").sum()
    )
    .reset_index()
    .rename(
        columns={
            "Capacity_Status":
            "Critical_Capacity_Alerts"
        }
    )
)

In [12]:
# Inventory Alert
inventory_alert_weekly = (
    inventory_risk
    .groupby("Order_Date")
    ["Inventory_Alert"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "Inventory_Alert":
            "Critical_Inventory_Alerts"
        }
    )
)

In [13]:
# Service Alert
service_alert_weekly = (
    service_risk
    .groupby("Order_Date")
    ["Service_Alert"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "Service_Alert":
            "High_Service_Risk_Events"
        }
    )
)

In [15]:
# Avg Capacity Utilization
weekly_utilization = (
    capacity_utilization
    .groupby("Order_Date")
    ["Utilization"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "Utilization":
            "Avg_Capacity_Utilization"
        }
    )
)

In [16]:
# merge
executive_dashboard_summary = (
    weekly_utilization
    .merge(
        capacity_alert_weekly,
        on="Order_Date",
        how="left"
    )
    .merge(
        inventory_alert_weekly,
        on="Order_Date",
        how="left"
    )
    .merge(
        service_alert_weekly,
        on="Order_Date",
        how="left"
    )
)

In [17]:
# Executive Risk Score
# Definition: Composite operational risk score

executive_dashboard_summary["Executive_Risk_Score"] = (
    executive_dashboard_summary[
        "Critical_Capacity_Alerts"
    ]
    +
    executive_dashboard_summary[
        "Critical_Inventory_Alerts"
    ]
    +
    executive_dashboard_summary[
        "High_Service_Risk_Events"
    ]
)

In [18]:
executive_dashboard_summary.head()

,Order_Date,Avg_Capacity_Utilization,Critical_Capacity_Alerts,Critical_Inventory_Alerts,High_Service_Risk_Events,Executive_Risk_Score
0,2015-01-04,0.476222,0,0,0,0
1,2015-01-11,0.800468,0,0,0,0
2,2015-01-18,0.878544,0,0,0,0
3,2015-01-25,0.864899,0,0,0,0
4,2015-02-01,0.846401,0,0,0,0


In [ ]:
#executive_dashboard_summary.to_csv(
#    "../data/processed/executive_dashboard_summary.csv",
#    index=False)

In [20]:
dashboard_root_cause_summary = (
    top_product_driver[
        [
            "Plant_ID",
            "Product Card Id",
            "Product Name",
            "Order_Date",
            "Weekly_Demand",
            "Demand_Rank",
            "Demand_Contribution",
            "Demand_Change",
            "Demand_Growth"
        ]
    ]
    .copy()
)

dashboard_root_cause_summary.to_csv(
    "../data/processed/dashboard_root_cause_summary.csv",
    index=False
)